<a href="https://colab.research.google.com/github/abhsrivastava/ActorGuice/blob/master/Publish_Model_to_HF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# In this note book we will deploy the model we fine tuned to hugging face


In [5]:
# install the dependencies

%pip install -q --upgrade transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 99.0 MB/s eta 0:00:00


In [3]:
import os
import torch

from pathlib import Path

from huggingface_hub import (
    HfApi,
    login,
    whoami
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

print("Publication tools imported successfully!")

Publication tools imported successfully!


In [5]:
from google.colab import drive
import os

drive.mount("/content/drive")

saved_model_directory = (
    "/content/drive/MyDrive/hugging-face-models/"
    "modernbert-bug-severity-final"
)

if os.path.isdir(saved_model_directory):
    print("Saved model directory found!")
    print(saved_model_directory)
else:
    raise FileNotFoundError(
        f"Could not find: {saved_model_directory}"
    )

Mounted at /content/drive
Saved model directory found!
/content/drive/MyDrive/hugging-face-models/modernbert-bug-severity-final


In [7]:
# Reload the models to check if it still can be loaded

release_tokenizer = AutoTokenizer.from_pretrained(
    saved_model_directory
)

release_model = AutoModelForSequenceClassification.from_pretrained(
    saved_model_directory
)

print("Saved model loaded successfully!")

print("\nModel architecture:")
print(release_model.config.architectures)

print("\nNumber of labels:")
print(release_model.config.num_labels)

print("\nID-to-label mapping:")
print(release_model.config.id2label)

print("\nLabel-to-ID mapping:")
print(release_model.config.label2id)

Loading weights:   0%|          | 0/138 [00:04<?, ?it/s]

Saved model loaded successfully!

Model architecture:
['ModernBertForSequenceClassification']

Number of labels:
6

ID-to-label mapping:
{0: 'blocker', 1: 'critical', 2: 'major', 3: 'minor', 4: 'normal', 5: 'trivial'}

Label-to-ID mapping:
{'blocker': 0, 'critical': 1, 'major': 2, 'minor': 3, 'normal': 4, 'trivial': 5}


In [9]:
MAX_TOKEN_LENGTH = 128

release_test_descriptions = [
    "Application crashes immediately and all unsaved data is lost.",
    "The button text is slightly misaligned.",
    "Users cannot log in after installing the latest update."
]

release_inputs = release_tokenizer(
    release_test_descriptions,
    padding=True,
    truncation=True,
    max_length=MAX_TOKEN_LENGTH,
    return_tensors="pt"
)

release_model.eval()

with torch.no_grad():
    release_outputs = release_model(**release_inputs)

release_probabilities = torch.softmax(
    release_outputs.logits,
    dim=-1
)

release_prediction_ids = torch.argmax(
    release_probabilities,
    dim=-1
)

for index, description in enumerate(release_test_descriptions):
    prediction_id = release_prediction_ids[index].item()

    severity_name = release_model.config.id2label[
        prediction_id
    ]

    confidence = release_probabilities[
        index,
        prediction_id
    ].item()

    print("\nBug description:")
    print(description)

    print("Predicted severity:", severity_name)
    print("Confidence:", round(confidence, 4))


Bug description:
Application crashes immediately and all unsaved data is lost.
Predicted severity: critical
Confidence: 0.6736

Bug description:
The button text is slightly misaligned.
Predicted severity: normal
Confidence: 0.5646

Bug description:
Users cannot log in after installing the latest update.
Predicted severity: normal
Confidence: 0.6935


In [15]:
from google.colab import drive

drive.mount("/content/drive")


import os

model_directory = (
    "/content/drive/MyDrive/modernbert-bug-severity-final"
)

if not os.path.exists(model_directory):
    raise FileNotFoundError(
        "The model release directory was not found: "
        + model_directory
    )

print("Release directory found!")

print("\nRelease files:")
for file_name in sorted(os.listdir(model_directory)):
    print(" -", file_name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Release directory found!

Release files:
 - config.json
 - model.safetensors
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin


In [18]:
import json
import os

model_directory = (
    "/content/drive/MyDrive/modernbert-bug-severity-final"
)

metrics_file_path = os.path.join(
    model_directory,
    "evaluation_metrics.json"
)

with open(metrics_file_path, "r") as metrics_file:
    release_metrics = json.load(metrics_file)

print("✅ Evaluation metrics loaded successfully!")

print("\n=== Saved Test Metrics ===")

for metric_name, metric_value in release_metrics["test_metrics"].items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

✅ Evaluation metrics loaded successfully!

=== Saved Test Metrics ===
test_loss: 0.5434
test_accuracy: 0.8710
test_macro_f1: 0.3295
test_macro_precision: 0.5992
test_macro_recall: 0.3136
test_runtime: 3.3278
test_samples_per_second: 300.5010
test_steps_per_second: 18.9320


In [19]:
# Login to HF

from google.colab import userdata

hugging_face_token = userdata.get("HF_TOKEN")

if hugging_face_token is None:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets."
    )

login(
    token=hugging_face_token,
    add_to_git_credential=False
)

print("Logged in to Hugging Face successfully!")

Logged in to Hugging Face successfully!


In [20]:
# Choose Hugging Face Repo Name

hugging_face_identity = whoami()

hugging_face_username = hugging_face_identity["name"]

repository_name = "modernbert-bug-severity"

repository_id = (
    f"{hugging_face_username}/{repository_name}"
)

print("Hugging Face username:")
print(hugging_face_username)

print("\nModel repository:")
print(repository_id)

print("\nFuture repository URL:")
print(f"https://huggingface.co/{repository_id}")

Hugging Face username:
abhishes

Model repository:
abhishes/modernbert-bug-severity

Future repository URL:
https://huggingface.co/abhishes/modernbert-bug-severity


In [22]:
# Create a Model Card

model_card_content = f"""---
language:
- en
library_name: transformers
pipeline_tag: text-classification
base_model: answerdotai/ModernBERT-base
datasets:
- AliArshad/Bugzilla_Eclipse_Bug_Reports_Dataset
metrics:
- accuracy
- f1
tags:
- modernbert
- bug-triage
- bug-severity
- sequence-classification
---

# ModernBERT Bug Severity Classifier

This model is a fully fine-tuned version of
`answerdotai/ModernBERT-base` for classifying short bug descriptions
into six severity levels:

- blocker
- critical
- major
- normal
- minor
- trivial

## Intended use

This model is an educational demonstration of automated bug-severity
classification.

It should not be used as the sole authority for production severity
decisions. High-impact predictions should be reviewed by a human.

## Training dataset

The model was trained using:

`AliArshad/Bugzilla_Eclipse_Bug_Reports_Dataset`

Only the `Short Description` field was used as the model input. The
`Severity Label` field was used as the target label.

## Base model

`answerdotai/ModernBERT-base`

## Training approach

This model was trained using full fine-tuning. It is not a LoRA or
adapter-only model.

Training included:

- Removing missing and empty descriptions
- Converting severity names into numeric labels
- Stratified training, validation, and test splits
- Batched tokenization
- Dynamic padding and attention masks
- Hugging Face Trainer
- Macro F1 checkpoint selection

## Evaluation

- Test accuracy: {release_metrics["test_metrics"]["test_accuracy"]}
- Test macro F1: {release_metrics["test_metrics"]["test_macro_f1"]}

Performance should also be examined separately for each severity using
the classification report and confusion matrix.

## Limitations

- Severity cannot always be determined from a short description alone.
- The training dataset may contain noisy or inconsistent labels.
- The severity classes are imbalanced.
- Historical bug reports may not represent current software practices.
- Softmax confidence is not guaranteed to be a calibrated probability.
- Human review is recommended for blocker and critical predictions.

## License status

The ModernBERT base model uses the Apache 2.0 license. The training
dataset's Hugging Face page does not currently declare a dataset
license. Confirm the applicable dataset and source-data terms before
making this fine-tuned model public.

## Example usage

Install the required library:

    pip install transformers torch

Run inference:

    from transformers import pipeline

    classifier = pipeline(
        "text-classification",
        model="{repository_id}",
        revision="v1.0.0"
    )

    bug_reports = [
        "Application crashes immediately and all data is lost.",
        "There is a spelling mistake in the documentation."
    ]

    results = classifier(bug_reports)

    print(results)
"""

model_card_path = (
    Path(saved_model_directory) / "README.md"
)

model_card_path.write_text(
    model_card_content,
    encoding="utf-8"
)

print("Model card created:")
print(model_card_path)

Model card created:
/content/drive/MyDrive/hugging-face-models/modernbert-bug-severity-final/README.md


In [23]:
# Inspect the model card

print(model_card_path.read_text(encoding="utf-8"))

---
language:
- en
library_name: transformers
pipeline_tag: text-classification
base_model: answerdotai/ModernBERT-base
datasets:
- AliArshad/Bugzilla_Eclipse_Bug_Reports_Dataset
metrics:
- accuracy
- f1
tags:
- modernbert
- bug-triage
- bug-severity
- sequence-classification
---

# ModernBERT Bug Severity Classifier

This model is a fully fine-tuned version of
`answerdotai/ModernBERT-base` for classifying short bug descriptions
into six severity levels:

- blocker
- critical
- major
- normal
- minor
- trivial

## Intended use

This model is an educational demonstration of automated bug-severity
classification.

It should not be used as the sole authority for production severity
decisions. High-impact predictions should be reviewed by a human.

## Training dataset

The model was trained using:

`AliArshad/Bugzilla_Eclipse_Bug_Reports_Dataset`

Only the `Short Description` field was used as the model input. The
`Severity Label` field was used as the target label.

## Base model

`answer

In [24]:
# Confirm the final contents to be uploaded

print("Files that will be uploaded:")

for file_name in sorted(
    os.listdir(saved_model_directory)
):
    file_path = os.path.join(
        saved_model_directory,
        file_name
    )

    file_size_in_megabytes = (
        os.path.getsize(file_path) / (1024 * 1024)
    )

    print(
        f"{file_name}: "
        f"{file_size_in_megabytes:.2f} MB"
    )

Files that will be uploaded:
README.md: 0.00 MB
config.json: 0.00 MB
model.safetensors: 570.73 MB
tokenizer.json: 3.42 MB
tokenizer_config.json: 0.00 MB
training_args.bin: 0.00 MB


In [25]:
# Create a private repo

hub_api = HfApi()

repository_url = hub_api.create_repo(
    repo_id=repository_id,
    repo_type="model",
    private=True,
    exist_ok=True
)

print("Private repository created:")
print(repository_url)

Private repository created:
https://huggingface.co/abhishes/modernbert-bug-severity


In [26]:
# Upload existing directory

upload_result = hub_api.upload_folder(
    folder_path=saved_model_directory,
    repo_id=repository_id,
    repo_type="model",
    commit_message="Upload initial bug severity model"
)

print("Upload completed!")

print("\nCommit URL:")
print(upload_result.commit_url)

Upload completed!

Commit URL:
https://huggingface.co/abhishes/modernbert-bug-severity/commit/72e99043e83d1fa291b005290ed1a022d2906f8a


In [27]:
# Load from Hugging Face

hub_tokenizer = AutoTokenizer.from_pretrained(
    repository_id,
    token=hugging_face_token
)

hub_model = AutoModelForSequenceClassification.from_pretrained(
    repository_id,
    token=hugging_face_token
)

print("Model downloaded from Hugging Face!")

print("\nLabels:")
print(hub_model.config.id2label)

config.json:   0%|          | 0.00/2.25k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  598MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Model downloaded from Hugging Face!

Labels:
{0: 'blocker', 1: 'critical', 2: 'major', 3: 'minor', 4: 'normal', 5: 'trivial'}


In [28]:
# Run Local infrence before making public

hub_test_descriptions = [
    "Application crashes and destroys unsaved work.",
    "The settings icon is one pixel too far to the left.",
    "Authentication fails for every user after the update."
]

hub_inputs = hub_tokenizer(
    hub_test_descriptions,
    padding=True,
    truncation=True,
    max_length=MAX_TOKEN_LENGTH,
    return_tensors="pt"
)

hub_model.eval()

with torch.no_grad():
    hub_outputs = hub_model(**hub_inputs)

hub_probabilities = torch.softmax(
    hub_outputs.logits,
    dim=-1
)

hub_prediction_ids = torch.argmax(
    hub_probabilities,
    dim=-1
)

for index, description in enumerate(hub_test_descriptions):
    prediction_id = hub_prediction_ids[index].item()

    severity_name = hub_model.config.id2label[
        prediction_id
    ]

    confidence = hub_probabilities[
        index,
        prediction_id
    ].item()

    print("\nBug description:")
    print(description)

    print("Predicted severity:", severity_name)
    print("Confidence:", round(confidence, 4))


Bug description:
Application crashes and destroys unsaved work.
Predicted severity: critical
Confidence: 0.5809

Bug description:
The settings icon is one pixel too far to the left.
Predicted severity: normal
Confidence: 0.9183

Bug description:
Authentication fails for every user after the update.
Predicted severity: normal
Confidence: 0.4556


In [29]:
# Create first version tag

release_tag = "v1.0.0"

hub_api.create_tag(
    repo_id=repository_id,
    repo_type="model",
    tag=release_tag,
    revision="main",
    tag_message="Initial bug severity classifier release",
    exist_ok=True
)

print("Release tag created:")
print(release_tag)

print("\nTagged release URL:")
print(
    f"https://huggingface.co/"
    f"{repository_id}/tree/{release_tag}"
)

Release tag created:
v1.0.0

Tagged release URL:
https://huggingface.co/abhishes/modernbert-bug-severity/tree/v1.0.0


In [30]:
# Verify the tagged release

tagged_model = AutoModelForSequenceClassification.from_pretrained(
    repository_id,
    revision="v1.0.0",
    token=hugging_face_token
)

tagged_tokenizer = AutoTokenizer.from_pretrained(
    repository_id,
    revision="v1.0.0",
    token=hugging_face_token
)

print("Tagged v1.0.0 release loaded successfully!")
print(tagged_model.config.id2label)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Tagged v1.0.0 release loaded successfully!
{0: 'blocker', 1: 'critical', 2: 'major', 3: 'minor', 4: 'normal', 5: 'trivial'}


In [31]:
# Make the repo public

hub_api.update_repo_settings(
    repo_id=repository_id,
    repo_type="model",
    private=False
)

print("The model repository is now public!")

print(
    f"https://huggingface.co/{repository_id}"
)

The model repository is now public!
https://huggingface.co/abhishes/modernbert-bug-severity
